# Travelling salesperson — a world tour

Visit every city once and return to the start, minimizing total distance. We build a great-circle (**haversine**) distance matrix with `ortidy.distance_matrix`, then solve a single-vehicle route. The result is an **edge-flow** frame with route features. See https://developers.google.com/optimization/routing/tsp

In [ ]:
import pandas as pd
import plotly.express as px

from ortidy import distance_matrix, solve_routing

cities = pd.DataFrame(
    {
        "city": [
            "Tokyo", "Delhi", "Cairo", "Moscow", "London", "Paris",
            "New York", "Sao Paulo", "Lagos", "Sydney", "Beijing",
            "Los Angeles",
        ],
        "lat": [
            35.68, 28.61, 30.04, 55.76, 51.51, 48.86,
            40.71, -23.55, 6.52, -33.87, 39.90, 34.05,
        ],
        "lon": [
            139.69, 77.21, 31.24, 37.62, -0.13, 2.35,
            -74.01, -46.63, 3.38, 151.21, 116.41, -118.24,
        ],
    }
)
cities

Build the haversine matrix (labelled by city) and solve, starting in Tokyo. `result.objective` is the total tour distance in kilometres.

In [ ]:
matrix = distance_matrix(cities, method="haversine", id_column="city")
result = solve_routing(matrix, starting_point="Tokyo")

print(result.status, "| total distance (km):", round(result.objective))
route = result.frame
route

Plot the tour on a globe (no geopandas required — Plotly maps lat/lon directly).

In [ ]:
plot = route.merge(cities, left_on="departure", right_on="city")
fig = px.line_geo(
    plot,
    lat="lat",
    lon="lon",
    projection="orthographic",
    hover_name="departure",
    hover_data=["destination", "distance", "tripsSinceStart", "distanceSinceStart"],
    title="Tour of the World",
    height=700,
)
fig